In [10]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\civil\qanun-madani.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\civil\qanun_madani_articles.tsv"

LAW_CODE = "qanun_madani"
LAW_NAME = "قانون مدنی"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return text.translate(trans_table)


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن و نرمال‌سازی اولیه
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط: هر سطح از سلسله‌مراتب باید در ابتدای خط جدید باشد
raw = re.sub(r"(?<!^)(جلد\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(کتاب\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(باب\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(فصل\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(مبحث\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(فقره\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(ماده\s*‌?\s*\d+\s*[–\-ـ]\s*)", r"\n\1", raw, flags=re.MULTILINE)

lines = raw.splitlines()

# 3) الگوهای شش سطح + ماده
volume_pattern = re.compile(r"^\s*جلد\s+(.+)$")
book_pattern = re.compile(r"^\s*کتاب\s+(.+)$")
bab_pattern = re.compile(r"^\s*باب\s+(.+)$")
chapter_pattern = re.compile(r"^\s*فصل\s+(.+)$")
section_pattern = re.compile(r"^\s*مبحث\s+(.+)$")
paragraph_pattern = re.compile(r"^\s*فقره\s+(.+)$")
article_pattern = re.compile(
    r"^\s*"
    r"(?:ماده|مواد)\s*"                           # ماده / مواد
    r"(?:‌|\s)*"                                   # انواع فاصله
    r"("                                           # شروع گروه شماره‌ها
        r"\d+"                                     # 1306
        r"(?:\s*تا\s*\d+)?"                        # تا 1308
        r"(?:\s*و\s*\d+)*"                         # و 1311
    r")"
    r"(?:\s*مکرر)?"                                # مکرر
    r"(?:\s*\([^)]+\))?"                           # پرانتز اختیاری
    r"\s*"
    r"[-–—ـ:]?"                                    # انواع خط‌تیره یا دونقطه
    r"\s*"
    r"(.*)$",                                      # متن ماده (ممکن است [...] باشد)
    re.UNICODE
)


# متغیرهای نگهداری سلسله‌مراتب فعلی
current_volume = ""
current_book = ""
current_bab = ""
current_chapter = ""
current_section = ""
current_paragraph = ""

articles = []

current_article_num = None
current_article_text = ""


for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) تشخیص جلد
    m_vol = volume_pattern.match(stripped)
    if m_vol:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_volume = normalize_persian(stripped)
        # ریست سطوح پایین‌تر
        current_book = ""
        current_bab = ""
        current_chapter = ""
        current_section = ""
        current_paragraph = ""
        continue

    # 2) تشخیص کتاب
    m_book = book_pattern.match(stripped)
    if m_book:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_book = normalize_persian(stripped)
        current_bab = ""
        current_chapter = ""
        current_section = ""
        current_paragraph = ""
        continue

    # 3) تشخیص باب
    m_bab = bab_pattern.match(stripped)
    if m_bab:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_bab = normalize_persian(stripped)
        current_chapter = ""
        current_section = ""
        current_paragraph = ""
        continue

    # 4) تشخیص فصل
    m_chap = chapter_pattern.match(stripped)
    if m_chap:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_chapter = normalize_persian(stripped)
        current_section = ""
        current_paragraph = ""
        continue

    # 5) تشخیص مبحث
    m_sec = section_pattern.match(stripped)
    if m_sec:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_section = normalize_persian(stripped)
        current_paragraph = ""
        continue

    # 6) تشخیص فقره
    m_para = paragraph_pattern.match(stripped)
    if m_para:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_paragraph = normalize_persian(stripped)
        continue

    # 7) تشخیص شروع ماده
    m_art = article_pattern.match(stripped)
    if m_art:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "volume": current_volume,
                    "book": current_book,
                    "bab": current_bab,
                    "chapter": current_chapter,
                    "section": current_section,
                    "paragraph": current_paragraph,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )

        num_str = m_art.group(1)
        rest_text = m_art.group(2).strip()

        try:
            current_article_num = int(num_str)
        except ValueError:
            current_article_num = None

        if rest_text:
            current_article_text = f"ماده {num_str}- {rest_text}"
        else:
            current_article_text = f"ماده {num_str}-"
        continue

    # 8) ادامهٔ متن مادهٔ جاری
    if current_article_num is not None:
        current_article_text += " " + stripped

# 9) بستن آخرین ماده
if current_article_num is not None and current_article_text:
    one_line = normalize_persian(current_article_text)
    articles.append(
        {
            "law_code": LAW_CODE,
            "law_name": LAW_NAME,
            "volume": current_volume,
            "book": current_book,
            "bab": current_bab,
            "chapter": current_chapter,
            "section": current_section,
            "paragraph": current_paragraph,
            "article_number": current_article_num,
            "text": one_line,
        }
    )

# 10) نوشتن خروجی TSV
fieldnames = [
    "law_code",
    "law_name",
    "volume",
    "book",
    "bab",
    "chapter",
    "section",
    "paragraph",
    "article_number",
    "text",
]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون مدنی تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون مدنی تمام شد.
✓ تعداد مواد استخراج شده: 1340
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\civil\qanun_madani_articles.tsv
